# client

> Client for interacting with the Fewsats API

In [ ]:
#| default_exp core

In [ ]:
#| export
from fastcore.utils import *
import os
import httpx
from typing import Dict, Any, List

In [ ]:
#| hide 
from dotenv import load_dotenv
from fastcore.test import *

In [ ]:
#| hide
load_dotenv()

True

The `Client` class handles authentication and provides the foundation for our API interactions.

In [ ]:
#| export
class Client:
    "Client for interacting with the Fewsats API"
    def __init__(self,
                 api_key: str = None, # The API key for the Fewsats account
                 base_url: str = "https://hub-5n97k.ondigitalocean.app"): # The Fewsats API base URL
        self.api_key = api_key or os.environ.get("FEWSATS_API_KEY")
        if not self.api_key:
            raise ValueError("The api_key client option must be set either by passing api_key to the client or by setting the FEWSATS_API_KEY environment variable")
        self.base_url = base_url
        self._httpx_client = httpx.Client()
        self._httpx_client.headers.update({"Authorization": f"Token {self.api_key}"})


In [ ]:
k = os.getenv("FEWSATS_API_KEY")
fs = Client(api_key=k)

test_eq(fs.api_key, k)
test_eq(fs._httpx_client.headers["Authorization"], f"Token {k}")

In [ ]:
#| export
@patch
def _request(self: Client, 
             method: str, # The HTTP method to use
             path: str, # The path to request
             **kwargs) -> Dict[str, Any]:
    "Makes an authenticated request to Fewsats API"
    url = f"{self.base_url}/{path}"
    return  self._httpx_client.request(method, url, **kwargs)

In [ ]:
# r  = fs._request("GET", "v0/stripe/payment-methods")
r  = fs._request("GET", "v0/users/me")
test_eq(r.status_code, 200)

In [ ]:
#| export

@patch
def me(self: Client):
    "Retrieve the user's info."
    r = self._request("GET", "v0/users/me")
    r.raise_for_status()
    return r.json()

In [ ]:
fs.me()

{'name': 'Fewsats',
 'last_name': 'Tester',
 'email': 'test@fewsats.com',
 'billing_info': None,
 'id': 15,
 'created_at': '2024-12-18T18:19:00.531Z'}

In [ ]:
#| export 

@patch
def balance(self: Client):
    "Retrieve the balance of the user's wallet."
    r = self._request("GET", "v0/wallets")
    r.raise_for_status()
    return r.json()

In [ ]:
fs.balance()

[{'id': 15, 'balance': 0, 'currency': 'usd'}]

## List Payment Methods

Retrieve the user's payment methods. Useful for checking which card will be used for purchases.

In [ ]:
#| export
@patch
def get_payment_methods(self: Client) -> List[Dict[str, Any]]:
    "Retrieve the user's payment methods, raises an exception for error status codes."
    r = self._request("GET", "v0/stripe/payment-methods")
    r.raise_for_status()
    return r.json()

In [ ]:

pm = fs.get_payment_methods()
pm

[]

In [ ]:
assert isinstance(pm, list)

## Simulate a Purchase

Simulate a purchase and return the resulting state. Useful, for example, to check if a CC charge is needed or the purchase will use the balance.

In [ ]:
#| export

@patch
def simulate_payment(self: Client,
                    amount: str): # The amount in USD cents
    "Simulates a purchase, raises an exception for error status codes."
    assert amount.isdigit()
    return self._request("POST", "v0/l402/preview/purchase/amount", json={"amount_usd": amount})


In [ ]:
p = fs.simulate_payment(amount="300") # 3.00 USD
p.json()


{'invoice': {'description': 'USD amount preview',
  'amount_usd': 300,
  'amount_btc': 0,
  'macaroon': '',
  'invoice': ''},
 'transaction': {'current_balance': 0,
  'balance_to_apply': 0,
  'amount_to_charge': 2000,
  'final_balance': 1700},
 'already_purchased': False,
 'purchase': None}

In [ ]:
#| hide
test_eq(p.status_code, 200)

## Pay a lightning invoice

Pay a lightning invoice. This is a low level method that should not used by most users. This method will use the default payment method if a charge is needed.

In [ ]:
#| export

@patch
def _pay_ln(self: Client,
         ln_invoice: str, # The Lightning Network invoice to pay
         description: str, # Short payment description
         l402_url: str = ""): # L402 URL of the resource
     "Pay an invoice, raises an exception for error status codes."
     p = {"invoice": ln_invoice, "description": description, "l402_url": l402_url, "macaroon":""}
     return self._request("POST", "v0/l402/purchases/direct", json=p)  

In [ ]:

pr = fs._pay_ln(ln_invoice="lnbc90n1pnk9ukupp57wcf8q2wacl5h986zch6alc24zwj8dhf42wk34dw2cuu44k5mauqdq6xysyxun9v35hggzsv93kkct8v5cqzpgxqrzpnrzjqwghf7zxvfkxq5a6sr65g0gdkv768p83mhsnt0msszapamzx2qvuxqqqqz99gpz55yqqqqqqqqqqqqqq9qrzjq25carzepgd4vqsyn44jrk85ezrpju92xyrk9apw4cdjh6yrwt5jgqqqqz99gpz55yqqqqqqqqqqqqqq9qsp564vc9ac6g6hshtxt9j4h7nh46sv9u566dlcrzug9rnvzplvgwq9s9qxpqysgqr4hhczsztdv625f6mzh2dc9u353nhtnpwcha4tp6kq2ztlqkw0ysegv6zwxj3tsfk447pyq90pszg26m9l5u9wc3xsjzrywkzvafm7gp32ap6k", description="1 credit in stock.l402.org", l402_url="https://stock.l402.org/ticker/GOOGL")
pr.json()

{'detail': 'An error occurred while paying the L402 invoice.'}

In [ ]:
#| hide
# we can't auto test this because ln invoices can only be paid once
# test_eq(pr.status_code, 200)

In [ ]:
#| export
@patch
def pay(self: Client,
        purl: str, # payment endpoint URL
        oid: str, # offer ID
        pct: str): # payment context token
    "Pay an invoice, raises an exception for error status codes."
    p = {"payment_request_url": purl, "offer_id": oid, "payment_context_token": pct}
    return self._request("POST", "v0/l402/purchases/from-offer", json=p)  

In [ ]:
# Example offer from stock.l402.org
oid = 'offer_c668e0c0'
purl = 'https://stock.l402.org/l402/payment-request'
pct='edb53dec-28f5-4cbb-924a-20e9003c20e1'

r = fs.pay(purl, oid, pct)
r

<Response [200 OK]>

In [ ]:
r.json()

{'id': 176,
 'created_at': '2024-12-18T19:05:28.401Z',
 'l402_url': 'https://stock.l402.org/l402/payment-request',
 'macaroon': '',
 'invoice': 'lnbc90n1pnkx88cpp50y774fz8nsnpukdvy7ky378mjtj5w7mhlcc2va4pnfv502xzds6qdq6xysyxun9v35hggzsv93kkct8v5cqzpgxqrzpnrzjqwghf7zxvfkxq5a6sr65g0gdkv768p83mhsnt0msszapamzx2qvuxqqqqz99gpz55yqqqqqqqqqqqqqq9qrzjq25carzepgd4vqsyn44jrk85ezrpju92xyrk9apw4cdjh6yrwt5jgqqqqz99gpz55yqqqqqqqqqqqqqq9qsp5su6ya76s7zt7d5qqwgjksq3f0w5wgnmfaueryagvyn37fjjcz6qq9qxpqysgq9tfztf0c0tksu43qk3anmkjedmr8uu5rss8fq0ah6k6n6lt7attrdjmns2m4f2ad7nh8gqqwa9es2sefwnn702p54x4glmzn7j3crmqqu95kk7',
 'preimage': 'b580894239ee07b740578210e44bd3a2876c1ee61fe46b201194bd3c31cde527',
 'amount': 0,
 'currency': 'usd',
 'description': '1 Credit Package'}

Both the preview and purchase methods automatically use the default payment method if a charge is needed. This client provides a straightforward way to interact with the Fewsats API, making it easy for developers to integrate Fewsats functionality into their applications.

In [ ]:
#|hide
from nbdev.doclinks import nbdev_export
nbdev_export()